### Cell 1 Configuration

In [ ]:
import os
import torch

class Config:
    CSV_TRAIN = "train.csv"
    IMG_DIR_TRAIN = "data/train"
    
    # Model Settings
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_WORKERS = 4  
    
    # Hyperparameters
    EPOCHS = 20
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    VAL_SPLIT_RATIO = 0.2
    RANDOM_SEED = 42
    
    # Hardware Device Detection
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {Config.DEVICE}")
if Config.DEVICE.type == 'cuda':
    print(f"🔥 GPU Model: {torch.cuda.get_device_name(0)}")

Using Device: cuda
🔥 GPU Model: NVIDIA GeForce RTX 3050 6GB Laptop GPU


### Dataset and DataLoader

In [ ]:
# CELL 2: DATASET & DATALOADERS (Using ImageFolder)

import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

# This resizes the images, applies augmentation, and converts them to tensors.
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load the Dataset
# This automatically handles the subfolders: data/train/bright, data/train/dark, etc.
full_dataset = ImageFolder(root="data/train", transform=train_transforms)

# 3. Split into 80% train, 20% validation
val_size = int(len(full_dataset) * Config.VAL_SPLIT_RATIO)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# 4. Prepare Dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=Config.NUM_WORKERS, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=Config.NUM_WORKERS, 
    pin_memory=True
)

print(f"✅ Loaded {len(full_dataset)} images from classes: {full_dataset.class_to_idx}")

✅ Loaded 1500 images from classes: {'bright': 0, 'dark': 1, 'normal': 2}


### CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

In [13]:
from dataset import IlluminationDataset, train_transforms, val_transforms
import torch.nn as nn
import torchvision.models as models

def build_illumination_model():
    # 1. Load the pre-trained ResNet18 model
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    # 2. Freeze all layers initially
    for param in model.parameters():
        param.requires_grad = False

    # 3. Unfreeze Layer 4 (The final convolutional block)
    # This allows the network to learn domain-specific illumination features
    for param in model.layer4.parameters():
        param.requires_grad = True

    # 4. Replace the Fully Connected (fc) classification head
    # Newly created layers automatically have requires_grad = True
    num_features = model.fc.in_features
    
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),            # Regularization to prevent overfitting
        nn.Linear(num_features, 3)    # 3 Output nodes: Dark(0), Normal(1), Bright(2)
    )

    return model

# Instantiate the model and move it to the GPU/CPU
model = build_illumination_model().to(Config.DEVICE)

# Verify the freezing strategy worked by counting parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"✅ Model Initialized on {Config.DEVICE}!")
print(f"🔒 Total Parameters: {total_params:,}")
print(f"🔥 Trainable Parameters: {trainable_params:,} (Layer 4 + FC Head)")

✅ Model Initialized on cuda!
🔒 Total Parameters: 11,178,051
🔥 Trainable Parameters: 8,395,267 (Layer 4 + FC Head)


#### Training on 8 million parameters

### CELL 4: TRAINING & VALIDATION ENGINE

In [14]:
import time
from tqdm import tqdm # Great for visual progress bars in VS Code

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    # Progress bar for the terminal/notebook
    loop = tqdm(dataloader, leave=False, desc="Training")
    
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        
        # 1. Forward Pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 2. Backward Pass & Optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 3. Track Metrics
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1) # Get the index of the highest probability
        correct_preds += torch.sum(preds == labels.data).item()
        total_samples += labels.size(0)
        
        # Update progress bar
        loop.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    with torch.no_grad(): # Disable gradient tracking for speed and memory
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_preds += torch.sum(preds == labels.data).item()
            total_samples += labels.size(0)
            
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

### CELL 5: MAIN EXECUTION LOOP

In [15]:
# 1. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()

# ONLY pass the trainable parameters to the optimizer
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = torch.optim.AdamW(
    trainable_params, 
    lr=Config.LEARNING_RATE, 
    weight_decay=Config.WEIGHT_DECAY
)

# 2. Initialize Tracking Variables
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("🔥 Starting Training...")
start_time = time.time()

# 3. The Main Epoch Loop
for epoch in range(Config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
    print("-" * 20)
    
    # Train and Validate
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, Config.DEVICE)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, Config.DEVICE)
    
    # Log metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    # 4. Checkpointing Strategy (Save ONLY if Accuracy improves)
    if val_acc > best_val_acc:
        print(f"⭐ Validation Accuracy improved from {best_val_acc:.4f} to {val_acc:.4f}. Saving checkpoint!")
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

time_elapsed = time.time() - start_time
print(f"\n✅ Training Complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_val_acc:.4f}")

🔥 Starting Training...

Epoch 1/20
--------------------


Train Loss: 1.2138 | Train Acc: 0.4042
Val Loss:   1.0467 | Val Acc:   0.4633
⭐ Validation Accuracy improved from 0.0000 to 0.4633. Saving checkpoint!

Epoch 2/20
--------------------


Train Loss: 0.9783 | Train Acc: 0.5258
Val Loss:   1.0580 | Val Acc:   0.4500

Epoch 3/20
--------------------


Train Loss: 0.8842 | Train Acc: 0.5842
Val Loss:   1.1158 | Val Acc:   0.4633

Epoch 4/20
--------------------


Train Loss: 0.7656 | Train Acc: 0.6642
Val Loss:   1.1289 | Val Acc:   0.4867
⭐ Validation Accuracy improved from 0.4633 to 0.4867. Saving checkpoint!

Epoch 5/20
--------------------


...

Train Loss: 0.1232 | Train Acc: 0.9592
Val Loss:   1.8074 | Val Acc:   0.4700

Epoch 17/20
--------------------


Train Loss: 0.0951 | Train Acc: 0.9683
Val Loss:   1.9151 | Val Acc:   0.4867

Epoch 18/20
--------------------


Train Loss: 0.0902 | Train Acc: 0.9700
Val Loss:   1.8620 | Val Acc:   0.4933
⭐ Validation Accuracy improved from 0.4900 to 0.4933. Saving checkpoint!

Epoch 19/20
--------------------


Train Loss: 0.0777 | Train Acc: 0.9767
Val Loss:   1.8039 | Val Acc:   0.4533

Epoch 20/20
--------------------


Train Loss: 0.0754 | Train Acc: 0.9783
Val Loss:   2.0104 | Val Acc:   0.4667

✅ Training Complete in 5m 56s
🏆 Best Validation Accuracy: 0.4933
